# SAM3 Auto-Label Pipeline

> Progressive walkthrough from basic repository inspection to an end-to-end pseudo-labeling export ready for fine-tuning.

In [ ]:
#| default_exp sam3_autolabel

In [ ]:
#| export
from __future__ import annotations

import json
import os
import subprocess
import sys
import textwrap
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from IPython.display import display
from PIL import Image
from pycocotools import mask as mask_utils

In [ ]:
WORKSPACE = Path("/workspace").resolve()
SAM3_REPO = (WORKSPACE / "sam3").resolve()
DATA_ROOT = (WORKSPACE / "data" / "autolabel").resolve()
RAW_DIR = DATA_ROOT / "images"
MASK_DIR = DATA_ROOT / "masks"

for path in (DATA_ROOT, RAW_DIR, MASK_DIR):
    path.mkdir(parents=True, exist_ok=True)

assert SAM3_REPO.exists(), "The SAM3 repository was not cloned into /workspace/sam3"

## 1. Inspect SAM3 segmentation hooks

We start by confirming the repository is present locally and by locating segmentation-specific code paths using ripgrep. This keeps the "simple first" rule intact while giving us concrete anchors for later sections.

In [ ]:
#| export
def ensure_repo(path: Path) -> Path:
    """Validate that ``path`` exists and looks like a git repo."""
    path = Path(path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"Expected repo at {path}")
    if not (path / ".git").exists():
        raise RuntimeError(f"{path} does not look like a git repository")
    return path


#| export
def search_repo(repo_path: Path, pattern: str, max_hits: int = 10) -> List[str]:
    """Return up to ``max_hits`` ripgrep lines that mention ``pattern``."""
    repo_path = ensure_repo(repo_path)
    cmd = [
        "rg",
        "-n",
        "-i",
        pattern,
        str(repo_path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, check=False)
    lines = [ln for ln in result.stdout.strip().splitlines() if ln]
    if not lines:
        return []
    return lines[:max_hits]

In [ ]:
segmentation_refs = search_repo(SAM3_REPO, "segmentation head", max_hits=5)
print("Found", len(segmentation_refs), "matches")
print("\n".join(segmentation_refs))

## 2. Pull a tiny dataset

Simple comes next: grab a handful of medical images so every helper we write immediately touches real pixels.

In [ ]:
#| export
def load_sample_records(
    dataset_id: str = "nielsr/breast-cancer",
    split: str = "train",
    count: int = 4,
) -> List[Dict[str, Any]]:
    """Return ``count`` image records (with masks when available)."""
    ds = load_dataset(dataset_id, split=f"{split}[:{count}]")
    records: List[Dict[str, Any]] = []
    for idx, sample in enumerate(ds):
        image = sample["image"]
        if not isinstance(image, Image.Image):
            image = Image.fromarray(np.array(image))
        mask = sample.get("label")
        if mask is not None and not isinstance(mask, np.ndarray):
            mask = np.array(mask)
        records.append(
            {
                "image": image,
                "mask": mask,
                "id": f"{dataset_id.replace('/', '_')}_{split}_{idx:04d}",
            }
        )
    return records


#| export
def show_overlay(image: Image.Image, mask: Optional[np.ndarray], alpha: float = 0.4) -> None:
    """Quick visualization helper to keep us honest."""
    plt.figure(figsize=(3, 3))
    plt.imshow(image)
    if mask is not None:
        plt.imshow(
            np.ma.masked_where(mask == 0, mask),
            cmap="magma",
            alpha=alpha,
        )
    plt.axis("off")
    plt.show()

In [ ]:
sample_records = load_sample_records(count=3)
print(f"Loaded {len(sample_records)} records")
show_overlay(sample_records[0]["image"], sample_records[0]["mask"])

## 3. Configuration + optional SAM3 processor

Keep configuration explicit and guard heavy model builds behind an environment flag (`RUN_SAM3=1`). This preserves the "single responsibility" rule—helpers focus on file I/O or inference, never both.

In [ ]:
#| export
@dataclass
class AutoLabelConfig:
    repo_root: Path = SAM3_REPO
    output_root: Path = DATA_ROOT
    prompt_template: str = "breast tumor"
    enable_sam3: bool = os.environ.get("RUN_SAM3", "0") == "1"
    device: str = "cuda"


#| export
@dataclass
class AutoLabelOutput:
    image_path: Path
    prompt: str
    masks: List[Path]
    metadata: Dict[str, Any]


#| export
def init_sam3_processor(cfg: AutoLabelConfig):
    """Optionally build a real SAM3 processor (requires HF access)."""
    if not cfg.enable_sam3:
        print("Skipping SAM3 init (set RUN_SAM3=1 to enable the real model).")
        return None

    sys.path.insert(0, str(cfg.repo_root))
    try:
        from sam3.model_builder import build_sam3_image_model
        from sam3.model.sam3_image_processor import Sam3Processor

        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = build_sam3_image_model(
            device=device,
            eval_mode=True,
            load_from_HF=False,
            enable_segmentation=True,
            enable_inst_interactivity=False,
        )
        return Sam3Processor(model, device=device)
    except Exception as exc:  # pragma: no cover - informative guard
        print(f"Failed to build SAM3 processor: {exc}")
        return None

In [ ]:
cfg = AutoLabelConfig()
processor = init_sam3_processor(cfg)

## 4. Lightweight fallback masks

When the full SAM3 weights are unavailable we still want deterministic tests. A simple percentile threshold keeps the pipeline grounded in real pixels.

In [ ]:
#| export
def approximate_mask(image: Image.Image, percentile: float = 85.0) -> np.ndarray:
    """Binarize an image via percentile thresholding (works as a fallback mask)."""
    gray = np.array(image.convert("L"))
    thresh = np.percentile(gray, percentile)
    return (gray >= thresh).astype(np.uint8)


#| export
def mask_to_bbox(mask: np.ndarray) -> List[int]:
    """Compute an ``[x, y, w, h]`` bbox for ``mask``."""
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        return [0, 0, 0, 0]
    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()
    return [int(x_min), int(y_min), int(x_max - x_min), int(y_max - y_min)]

In [ ]:
fallback_mask = approximate_mask(sample_records[0]["image"])
show_overlay(sample_records[0]["image"], fallback_mask)
print("BBox:", mask_to_bbox(fallback_mask))

## 5. Auto-labeler class

Single-responsibility helpers persist data to disk, wrap SAM3 (or the fallback), and expose metadata for downstream fine-tuning.

In [ ]:
#| export
class AutoLabeler:
    """Persist pseudo labels produced by SAM3 or a deterministic fallback."""

    def __init__(self, cfg: AutoLabelConfig, processor=None):
        self.cfg = cfg
        self.processor = processor
        self.image_dir = cfg.output_root / "images"
        self.mask_dir = cfg.output_root / "masks"
        self.image_dir.mkdir(parents=True, exist_ok=True)
        self.mask_dir.mkdir(parents=True, exist_ok=True)
        self._counter = 0

    def _next_id(self) -> str:
        self._counter += 1
        return f"sample_{self._counter:05d}"

    def _run_sam3(self, image: Image.Image, prompt: str) -> Optional[np.ndarray]:
        if self.processor is None:
            return None
        state = self.processor.set_image(image)
        outputs = self.processor.set_text_prompt(prompt, state)
        masks = outputs.get("masks")
        if masks is None or len(masks) == 0:
            return None
        return (masks[0].detach().cpu().numpy() > 0.5).astype(np.uint8)

    def label_image(self, record: Dict[str, Any], prompt: Optional[str] = None) -> AutoLabelOutput:
        prompt = prompt or self.cfg.prompt_template
        image: Image.Image = record["image"]
        image_id: str = record.get("id") or self._next_id()
        image_path = self.image_dir / f"{image_id}.png"
        image.save(image_path)

        mask = self._run_sam3(image, prompt)
        if mask is None:
            if record.get("mask") is not None:
                mask = np.array(record["mask"]).astype(np.uint8)
            else:
                mask = approximate_mask(image)

        mask_path = self.mask_dir / f"{image_id}_mask.png"
        Image.fromarray(mask * 255).save(mask_path)

        metadata = {
            "image_id": image_id,
            "prompt": prompt,
            "bbox_xywh": mask_to_bbox(mask),
            "mask_pixels": int(mask.sum()),
            "height": int(mask.shape[0]),
            "width": int(mask.shape[1]),
        }

        return AutoLabelOutput(
            image_path=image_path,
            prompt=prompt,
            masks=[mask_path],
            metadata=metadata,
        )

In [ ]:
labeler = AutoLabeler(cfg, processor)
auto_outputs = [labeler.label_image(record) for record in sample_records]
auto_outputs[0].metadata

In [ ]:
def plot_labeled_example(output: AutoLabelOutput) -> None:
    image = Image.open(output.image_path)
    mask = np.array(Image.open(output.masks[0]).convert("L"))
    show_overlay(image, mask)
    print(output.metadata)

plot_labeled_example(auto_outputs[0])

## 6. Build a COCO-style manifest

With files on disk we can export a single JSON that SAM3's training configs already understand (`load_segmentation=True`).

In [ ]:
#| export
def mask_path_to_rle(mask_path: Path) -> Dict[str, Any]:
    mask_arr = np.array(Image.open(mask_path).convert("1"), dtype=np.uint8)
    rle = mask_utils.encode(np.asfortranarray(mask_arr))
    rle["counts"] = rle["counts"].decode("utf-8")
    return rle


#| export
def build_coco_dict(outputs: List[AutoLabelOutput], dataset_name: str = "autolabel") -> Dict[str, Any]:
    images, annotations = [], []
    categories = [
        {
            "id": 1,
            "name": dataset_name,
            "supercategory": "concept",
        }
    ]

    annotation_id = 1
    for image_id, out in enumerate(outputs, start=1):
        img = Image.open(out.image_path)
        width, height = img.size
        images.append(
            {
                "id": image_id,
                "file_name": out.image_path.name,
                "width": width,
                "height": height,
            }
        )
        for mask_path in out.masks:
            mask_arr = np.array(Image.open(mask_path).convert("1"), dtype=np.uint8)
            annotations.append(
                {
                    "id": annotation_id,
                    "image_id": image_id,
                    "category_id": 1,
                    "bbox": mask_to_bbox(mask_arr),
                    "area": int(mask_arr.sum()),
                    "segmentation": mask_path_to_rle(mask_path),
                    "iscrowd": 0,
                }
            )
            annotation_id += 1

    return {
        "images": images,
        "annotations": annotations,
        "categories": categories,
    }


#| export
def write_json(data: Dict[str, Any], path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(data, f)
    return path

In [ ]:
coco_dict = build_coco_dict(auto_outputs, dataset_name="breast-tumor")
annotations_path = write_json(coco_dict, DATA_ROOT / "autolabel_annotations.json")
annotations_path

## 7. Where to go next

- Flip `RUN_SAM3=1` and rerun Section 5 to record genuine SAM3 masks once you have the gated checkpoint.
- Point `sam3/train/configs/roboflow_v100_full_ft_100_images.yaml` at `DATA_ROOT` and set `scratch.enable_segmentation: True` for pixel-level losses.
- Track iterations by committing the generated JSON + PNG assets or pushing them to Hugging Face Datasets for reproducibility.
- Extend `AutoLabeler` with quality filters (score thresholds, minimum area) before exporting to avoid noisy pseudo labels.